# HW4P1: Decoder-Only Language Modeling

In this assignment, you will implement and train a decoder-only Transformer.
Complete each implementation section and run the test directly below it before
starting your experiments.


# Schedule:
- Early Submission (DUE 20 November 2026 @ 11:59PM EST)
- Final Submission (DUE 4 December 2026 @ 11:59PM EST)


# Setup

Create an `IDL-26Fall-HW4` folder and extract `hw4lib.zip` into it once, before starting P1. Place both notebooks beside `hw4lib/`, `mytorch/`, `tests/`, and the requirements files.

Use this project folder and the same environment for P1 and P2. Continue P2 with your completed P1 files and checkpoint.


## Local

Create one environment for both notebooks:

```bash
conda create -n hw4 python=3.13
conda activate hw4
pip install -r requirements.txt
```

Open the notebook from `IDL-26Fall-HW4/` and select the `hw4` environment as its kernel. The same installation includes the P2 dependencies.

Run implementation tests on CPU. Use a supported GPU for training.


## Colab

Upload or mount the project folder containing the shared code and both notebooks. Set its path in the cell below.

Install `requirements-colab.txt` for both P1 and P2. Keep Colab's installed Torch/CUDA stack.


In [ ]:
%cd /content/IDL-26Fall-HW4
%pip install -q -r requirements-colab.txt

import torch
print("PyTorch:", torch.__version__, "CUDA:", torch.version.cuda)


### Download the data

Open the [private Kaggle data link](https://kaggle.com/datasets/1c76b16eea78fc65068ec05055fa9b2d61e4f2d4f350920cf24b0e457770c4c1).
Run the cell below to download `mini_train`, `mini_val`, `train`, `val`, and `test`.
To use an extracted copy, set `manual_data_root` to its directory.
Keep the sharing link within the course.


In [ ]:
import json
import os
import shutil
import urllib.request
from pathlib import Path

# Optional: use an existing extracted copy.
manual_data_root = os.environ.get("HW4P1_DATA_ROOT", "")

if manual_data_root:
    data_root = str(Path(manual_data_root).expanduser().resolve())
else:
    shared_link = "https://kaggle.com/datasets/1c76b16eea78fc65068ec05055fa9b2d61e4f2d4f350920cf24b0e457770c4c1"
    download_dir = Path(os.environ.get("HW4P1_DOWNLOAD_DIR", "/content"))
    dataset_dir = download_dir / "hw4p1-data"

    def download_file(relative_path):
        destination = dataset_dir / relative_path
        destination.parent.mkdir(parents=True, exist_ok=True)
        request = urllib.request.Request(
            "https://api.kaggle.com/v1/datasets.DatasetApiService/DownloadDataset",
            data=json.dumps({
                "ownerSlug": "changpengyu",
                "datasetSlug": "f26-hw4p1-data",
                "datasetVersionNumber": 1,
                "hashLink": shared_link.rsplit("/", 1)[-1],
                "fileName": f"hw4p1-data/{relative_path}",
            }).encode(),
            headers={"Content-Type": "application/json"},
        )
        with urllib.request.urlopen(request, timeout=60) as response:
            with destination.open("wb") as output:
                shutil.copyfileobj(response, output)

    for partition in ("mini_train", "mini_val", "train", "val", "test"):
        download_file(f"{partition}/manifest.json")
        manifest = json.loads((dataset_dir / partition / "manifest.json").read_text())
        for shard in manifest["shards"]:
            download_file(f"{partition}/{shard['file']}")
        print("Downloaded:", partition)
    download_file("README.md")
    download_file("SOURCE_LICENSE.md")
    data_root = str(dataset_dir.resolve())

os.environ["HW4P1_DATA_ROOT"] = data_root
print("Data root:", data_root)


### Step 4: Move to the handout directory

You may need to repeat this after restarting the runtime.


In [ ]:
%cd /content/IDL-26Fall-HW4
!pwd
!ls


## Kaggle

1. Choose **Copy & Edit** on the course notebook or upload the handout. Keep editable files under `/kaggle/working`.
2. Select a GPU in Notebook Settings.
3. Sign in and open the [private data-sharing link](https://kaggle.com/datasets/1c76b16eea78fc65068ec05055fa9b2d61e4f2d4f350920cf24b0e457770c4c1). Attach **F26 HW4P1 Data** through **Add Input**. Keep the link within the course.
4. Set the handout directory below.


In [ ]:
%cd /kaggle/working/IDL-26Fall-HW4
!pwd
!ls
%pip install -q -r requirements-colab.txt


In [ ]:
import os
from pathlib import Path

# Optional: set the path yourself if you attached more than one copy.
manual_data_root = os.environ.get("HW4P1_DATA_ROOT", "")
if manual_data_root:
    data_root = str(Path(manual_data_root).expanduser().resolve())
else:
    manifest_path = next(Path("/kaggle/input").glob("**/mini_train/manifest.json"))
    data_root = str(manifest_path.parents[1])
os.environ["HW4P1_DATA_ROOT"] = data_root
print("Data root:", data_root)


## Delta

### Step 1: Setting Up Your Environment on Delta

❗️⚠️ For this homework, we are **providing a shared Conda environment and shared Dataset** for the entire class. Therefore, Delta users **do not need to manually install any packages or download the dataset**.

Follow these steps to set up the environment and start a Jupyter notebook on Delta:

- To run your notebook more efficiently on Delta, we need to use a **Jupyter Server** hosted on a compute node.

- You can use your prefered way of connecting to the Jupyter Server.  **The recommended way is to connect in VSCode.**
Follow the instructions below.

#### **Connect in VSCode**
SSH into Delta and navigate to your **Home directory** (`/u/<your_NCSA_username>`). Upload your notebook there, and then connect to the Jupyter Server from that directory.

#### **1.1 SSH into Delta**
1）Open VS Code and click on the `Extensions` icon in the left sidebar. Make sure the "**Remote - SSH**" extension is installed.

2）Open the command palette (**Shift+Command+P** on Mac, **Ctrl+Shift+P** on Windows). A search box will appear at the top center. Choose `"Remote-SSH: Add New SSH Host"`, then enter:

```bash
ssh <your_NCSA_username>@login.delta.ncsa.illinois.edu #change <your_NCSA_username> to your NCSA username
```

Next, choose `"/Users/<your_username>/.ssh/config"` as the config file. A dialog will appear in the bottom right saying "Host Added". Click `"Connect"`, and then enter your password.

(Note: After adding the host once, you can later use `"Remote-SSH: Connect to Host"` and select "login.delta.ncsa.illinois.edu" from the list.)

3）Once connected, click `"Explorer"` in the left sidebar > "Open Folder", and navigate to your home directory:
```bash
/u/<your_username>  #change <your_username> to your username
```

4）You can now drag your notebook files directly into the right-hand pane (your remote home directory), or upload them using `scp` into your folder.

> ❗️⚠️ The following steps should be executed in the **VSCode integrated terminal**.

#### **1.2 Navigate to Your Directory**
Make sure to use this `/u/<your_username>` as your working directory, since all subsequent operations (up to submission) are based on this path.
```bash
cd /u/<your_username> #change <your_username> to your username
```

#### **1.3 Request a Compute Node**
**Resource Configuration:** The command below requests computational resources on Delta using Slurm:

- `--account`: Specifies the allocation account to charge. Use the account assigned to you.
- `--partition`: Selects the GPU partition. **For the HW allocation, you must use A40 GPUs (`gpuA40x4`). Higher-tier GPUs are not permitted.**
- `--nodes=1`: Requests one compute node.
- `--ntasks=1`: Runs one task.
- `--gpus=1`: Requests one A40 GPU.
- `--cpus-per-task=8`: Allocates 8 CPU cores for data loading and preprocessing. Each A40 node has 64 CPU cores, with **16 cores per GPU as the standard resource allocation**.
- `--mem=32G`: Allocates 32 GB of system RAM (not GPU memory). Each A40 node has 256 GB RAM, with **62.5 GB per GPU as the standard resource allocation**.
- `--time=08:00:00`: Sets the maximum runtime to 8 hours. The `gpuA40x4` partition allows **up to 48 hours per job**.
- `--pty bash`: Opens an interactive Bash shell on the allocated compute node.

**Note:** The CPU and memory settings are recommended starting values for HW4P1 and HW4P2, not fixed requirements. You may adjust `--cpus-per-task`, `--mem`, and `--time` based on your workload. For a single A40 GPU, please stay within 16 CPU cores and 62.5 GB RAM to avoid additional resource charges. Larger resource requests may increase queue times and charges. GPU usage is charged based on actual allocated runtime, not the requested time limit. Please exit the interactive session when finished to release your resources.

```bash
srun --account=bjor-delta-gpu \
  --partition=gpuA40x4 \
  --nodes=1 \
  --ntasks=1 \
  --gpus=1 \
  --cpus-per-task=8 \
  --mem=32G \
  --time=08:00:00 \
  --pty bash
```

#### **1.4 Load the PyTorch/Conda Module**
```bash
module load pytorch-conda/2.8
```

#### **1.5 Activate the provided HW Environment**
```bash
conda deactivate # First, deactivate any existing Conda environment
conda activate /projects/bjor/shared/envs/IDLF26 && export PYTHONNOUSERSITE=1
```

#### **1.6 Start Jupyter Notebook**
Launch Jupyter Notebook:
```bash
jupyter notebook --no-browser --ip=0.0.0.0
```

Go to **Kernel** → **Select Another Kernel** → **Existing Jupyter Server**
   Enter the URL of the Jupyter Server:```http://{hostname}:{port}/tree?token={token}```
   
   *(Usually, this URL appears in the terminal output after you run `jupyter notebook --no-browser --ip=0.0.0.0`, in a line like:  “Jupyter Server is running at: http://...”)*

   - eg: `http://gpub052.delta.ncsa.illinois.edu:8888/tree?token=883c552b49942ef7bb15052ccd1578dc7ae82f365fa63f64`

> **Note**: Replace `{hostname}`, `{port}` and `{token}` with your actual values from the Jupyter output.

After launching the Jupyter notebook, you can run the cells directly inside the notebook — no need to use the terminal for the remaining steps.

### Step 2: Set Up the HW4P1 Repo/Project

Complete the TODOs in the HW4P1 handout and place the `hw4lib/`, `mytorch/`, and `tests/` folders under your HW4P1 project directory, keeping the original directory structure.

In [ ]:
#Make sure you are in your directory
!pwd #should be /u/<your_NCSA_username>, if not, uncomment the following line and replace with your actual username:
%cd /u/<your_NCSA_username>
#TODO: replace the "<your_NCSA_username>" to yours

In [ ]:
# Example: My preferred approach
import os
# Settings -> Developer Settings -> Personal Access Tokens -> Token (classic)
os.environ['GITHUB_TOKEN'] = "your_github_token_here"

GITHUB_USERNAME = "your_github_username_here"
REPO_NAME       = "your_github_repo_name_here"
TOKEN = os.environ.get("GITHUB_TOKEN")
repo_url        = f"https://{TOKEN}@github.com/{GITHUB_USERNAME}/{REPO_NAME}.git"
!git clone {repo_url}

In [ ]:
# To pull latest changes (Must be in the repo dir, use pwd/ls to verify)
!cd {REPO_NAME} && git pull

#### **Move to Project Directory**
- `NOTE`: You may have to repeat this on anytime you restart your runtime. You can do a `pwd` or `ls` to check if you are in the right directory.

In [ ]:
%cd <Your-HW4P1-Project-Directory-Path> 
#TODO: Change this to the path of your HW4P1 project directory. 

#### Set Data and Output Directories

❗️⚠️ The data used in this assignment is **already stored in a shared, read-only folder, so you do not need to manually download anything**.

Instead, just make sure to replace the dataset path in your notebook code with the correct path from the shared directory.

You can run the following block to explore the shared directory structure:

In [ ]:
import os
from pathlib import Path

os.environ["HW4P1_DATA_ROOT"] = "/work/hdd/bjor/hw4p1_data" # this is the path of the shared dataset
os.environ["HW4P1_OUTPUT_ROOT"] = "<Your-HW4P1-Output-Directory-Path>"

Path(os.environ["HW4P1_OUTPUT_ROOT"]).mkdir(
    parents=True, exist_ok=True
)

In [ ]:
import os
data_path = "/work/hdd/bjor/hw4p1_data" 
print("Files in shared hw4p1 dataset:", os.listdir(data_path))

In [ ]:
!tree -L 2 /work/hdd/bjor/hw4p1_data | head -n 20

# Imports

Run this notebook from the `IDL-26Fall-HW4/` folder containing `hw4lib/`.
If you extracted the handout ZIP, use the extracted folder.


In [ ]:
from pathlib import Path
import gc
import json
import os
import random
import shutil
import sys

import numpy as np
import torch
import yaml
from torch.utils.data import DataLoader

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "hw4lib").is_dir():
    PROJECT_ROOT = PROJECT_ROOT / "IDL-26Fall-HW4"
os.chdir(PROJECT_ROOT)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from hw4lib.data import GPT2Tokenizer, LMDataset
from hw4lib.data.window_sampler import EpochWindowSampler
from hw4lib.model import DecoderOnlyTransformer
from hw4lib.trainers import LMTrainer
from hw4lib.utils import create_optimizer, create_scheduler

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Project root:", PROJECT_ROOT)
print("Using device:", device)


# Implementations

Read the specifications in each source file before filling in the TODOs.
The tests use small CPU models; their parameter counts are not your experiment model.


## HW3P1 Prerequisites

Use your HW3P1 implementations of `Softmax`, `Linear`, `ScaledDotProductAttention`, and `MultiHeadAttention` in `mytorch/nn`.

A fully masked attention row must produce zero probabilities, output, and gradient contribution.


In [ ]:
!"{sys.executable}" -m tests.test_mytorch_softmax


In [ ]:
!"{sys.executable}" -m tests.test_mytorch_linear


In [ ]:
!"{sys.executable}" -m tests.test_mytorch_scaled_dot_product_attention


In [ ]:
!"{sys.executable}" -m tests.test_mytorch_multi_head_attention


## Grouped-Query and Multi-Query Attention

Complete `GroupedQueryAttention` in `mytorch/nn`. `MultiQueryAttention` is the
one-KV-head case and inherits your GQA implementation.


In [ ]:
!"{sys.executable}" -m tests.test_mytorch_grouped_query_attention


## GPT-2 Tokenizer and LM Dataset

The GPT-2 tokenizer and shard format are provided. Complete the fixed-window LM
dataset in `hw4lib/data/lm_dataset.py`.


In [ ]:
!"{sys.executable}" -m tests.test_gpt2_data


## Positional Encoding

Use your HW3P2 implementations of `PositionalEncoding` and `RotaryPositionalEncoding` in `hw4lib/model/positional_encoding.py`.


In [ ]:
!"{sys.executable}" -m tests.test_positional_encoding


In [ ]:
!"{sys.executable}" -m tests.test_rotary_positional_encoding


## Causal Self-Attention

Use your RoPE implementation to complete causal MHA/GQA/MQA in
`hw4lib/model/causal_self_attention.py`.


In [ ]:
!"{sys.executable}" -m tests.test_causal_self_attention


## KV Cache

Complete the compact KV-cache path in `CausalSelfAttention`. Store only the
KV heads and use the cached length as the RoPE position offset.


In [ ]:
!"{sys.executable}" -m tests.test_kv_cache


## Masks

Use your HW3P2 implementations of `PadMask` and `CausalMask` in `hw4lib/model/masks.py`.


In [ ]:
!"{sys.executable}" -m tests.test_mask_padding


In [ ]:
!"{sys.executable}" -m tests.test_mask_causal


## Self-Attention Layer

Complete `SelfAttentionLayer`, including its cached path.


In [ ]:
!"{sys.executable}" -m tests.test_sublayer_selfattention


## Feed-Forward Layer

Use your HW3P2 implementation of `FeedForwardLayer` in `hw4lib/model/sublayers.py`.


In [ ]:
!"{sys.executable}" -m tests.test_sublayer_feedforward


## Self-Attention Decoder Layer

Complete `SelfAttentionDecoderLayer`, including cache threading.


In [ ]:
!"{sys.executable}" -m tests.test_decoderlayer_selfattention


## Decoder-Only Transformer

Complete `DecoderOnlyTransformer`, including its per-layer cache.


In [ ]:
!"{sys.executable}" -m tests.test_transformer_decoder_only


## Greedy Decoding

Complete `generate_greedy` in `hw4lib/decoding/sequence_generator.py`.


In [ ]:
!"{sys.executable}" -m tests.test_decoding_greedy


## Sampling

Complete `_filter_logits` and `generate_sample` in
`hw4lib/decoding/sequence_generator.py`.


In [ ]:
!"{sys.executable}" -m tests.test_decoding_sampling


#### Optional: Beam Search

You may implement `generate_beam`. It is not required or graded.


In [ ]:
# Remove the leading # after implementing beam search.
# !"{sys.executable}" -m tests.test_decoding_beam


## Optimizer and Scheduler

The scheduler utility is provided. Run this check before testing trainer gradient
accumulation.


In [ ]:
!"{sys.executable}" -m tests.test_lr_scheduler_accumulation


## LM Trainer

Complete the TODOs in `hw4lib/trainers/lm_trainer.py`, including
gradient accumulation, validation, and generation.


In [ ]:
!"{sys.executable}" -m tests.test_lm_trainer_p1


## End-to-End Check

Run this after all tests above pass.


In [ ]:
!"{sys.executable}" -m tests.test_hw4p1_pipeline


# Experiments

Start with `mini_train` and `mini_val`. After the model trains correctly, update the
configuration for `train` and `val` and run your final experiment.


## Data

The setup section for your environment must set `HW4P1_DATA_ROOT` before you run
this cell.


In [ ]:
data_root_value = os.environ.get("HW4P1_DATA_ROOT")

DATA_ROOT = Path(data_root_value).expanduser().resolve()

print("Data root:", DATA_ROOT)


## Config

The values below are only a starting point for the mini data. Choose your final
architecture and training settings, then change the partitions to `train`, `val`,
and `test` before final training.

For shorter experiments, set `data.max_train_windows_per_epoch` to a positive
integer (for example, `20000`). Each epoch draws that many distinct training
windows, capped at the partition size. `null` uses the whole training partition.
The seed and epoch determine the draw; windows may recur across epochs.
Validation and test still use every window in their selected partitions.

With a budget, an epoch is a sampled pass, not a full corpus pass. Compare runs
using windows/tokens processed as well as epochs. Cosine and warmup durations
use this loader's optimizer-update count, including gradient accumulation.
Rebuild the loaders and scheduler after changing the budget. For an existing
checkpoint, keep the original budget and scheduler settings when resuming.


In [ ]:
%%writefile config.yaml

Name: "hw4p1-mini"

###### Dataset -----------------------------------------------------------------
data:
  root            : ""           # Filled from HW4P1_DATA_ROOT in the next cell
  train_partition : "mini_train" # Change to train for final training
  val_partition   : "mini_val"   # Change to val for final training
  test_partition  : "mini_val"   # Change to test for final evaluation
  context_length  : 128
  stride          : 128
  batch_size      : 32
  max_train_windows_per_epoch: null # Positive integer for a shorter sampled epoch
  num_workers     : 2             # Set to 0 when debugging dataloaders

###### Network Specs -----------------------------------------------------------
model: # Decoder-Only Transformer (HW4P1)
  num_layers      : 2
  d_model         : 128
  num_heads       : 4             # Query heads
  num_kv_heads    : 2             # MHA: num_heads, GQA: 2+, MQA: 1
  d_ff            : 512
  dropout         : 0.1
  layer_drop_rate : 0.0
  weight_tying    : True
  use_rope        : True
  rope_base       : 10000.0

###### Training ----------------------------------------------------------------
training:
  epochs                          : 2
  gradient_accumulation_steps     : 1
  seed                            : 11785
  label_smoothing                 : 0.0
  use_wandb                       : True   # Required for this assignment; keep True.
  wandb_project                   : "hw4p1"
  wandb_entity                    : ""     # W&B username or team; blank uses default
  wandb_run_id                    : "none" # Set this only when resuming a W&B run
  wandb_log_checkpoints           : True   # Large files; local checkpoints are always saved

###### Optimizer ---------------------------------------------------------------
optimizer:
  name         : "adamw"          # sgd, adam, adamw
  lr           : 0.0005
  weight_decay : 0.01

  param_groups: []

  layer_decay:
    enabled    : False
    decay_rate : 0.75

  sgd:
    momentum   : 0.9
    nesterov   : True
    dampening  : 0

  adam:
    betas      : [0.9, 0.999]
    eps        : 1.0e-8
    amsgrad    : False

  adamw:
    betas      : [0.9, 0.95]
    eps        : 1.0e-8
    amsgrad    : False

###### Scheduler ---------------------------------------------------------------
scheduler:
  name: "cosine"                  # reduce_lr, cosine, cosine_warm

  reduce_lr:
    mode           : "min"
    factor         : 0.1
    patience       : 2
    threshold      : 0.0001
    threshold_mode : "rel"
    cooldown       : 0
    min_lr         : 1.0e-7
    eps            : 1.0e-8

  cosine:
    T_max      : 2                 # Match training.epochs
    eta_min    : 1.0e-5
    last_epoch : -1

  cosine_warm:
    T_0        : 2
    T_mult     : 2
    eta_min    : 1.0e-5
    last_epoch : -1

  warmup:
    enabled      : False
    type         : "linear"
    epochs       : 1
    start_factor : 0.1
    end_factor   : 1.0


In [ ]:
CONFIG_PATH = PROJECT_ROOT / "config.yaml"
with CONFIG_PATH.open("r", encoding="utf-8") as config_file:
    config = yaml.safe_load(config_file)

config["data"]["root"] = str(DATA_ROOT)
seed = int(config["training"]["seed"])
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(seed)

CONFIG_PATH.write_text(
    yaml.safe_dump(config, sort_keys=False),
    encoding="utf-8",
)
print(CONFIG_PATH.read_text(encoding="utf-8"))


## Weights & Biases

W&B logging is required for this assignment. Keep training.use_wandb set to True. 


W&B is controlled by `training.use_wandb` in `config.yaml`. Store a secret named
`WANDB_API_KEY` in Colab or Kaggle, or export it in your local/Delta environment.
If no secret is found, the next cell uses a hidden prompt. Never paste the key into
the notebook source.


In [ ]:
import getpass

if not config["training"].get("use_wandb", False):
    raise RuntimeError(
        "W&B logging is required. Set training.use_wandb to True."
    )
if config["training"].get("use_wandb", False):
    try:
        import wandb
    except ImportError as exc:
        raise RuntimeError(
            "Install the requirements before enabling W&B."
        ) from exc

    wandb_key = os.environ.get("WANDB_API_KEY", "")

    if not wandb_key:
        try:
            from google.colab import userdata
            wandb_key = userdata.get("WANDB_API_KEY") or ""
        except Exception:
            pass

    if not wandb_key:
        try:
            from kaggle_secrets import UserSecretsClient
            wandb_key = UserSecretsClient().get_secret("WANDB_API_KEY") or ""
        except Exception:
            pass

    if not wandb_key:
        wandb_key = getpass.getpass("W&B API key: ")
    if not wandb_key:
        raise RuntimeError("WANDB_API_KEY is required when use_wandb is True.")

    os.environ["WANDB_API_KEY"] = wandb_key
    del wandb_key
    wandb.login()
    print("W&B project:", config["training"]["wandb_project"])
else:
    print("W&B logging is disabled in config.yaml.")


## Tokenizer and Datasets


In [ ]:
tokenizer = GPT2Tokenizer()
data_config = config["data"]

train_dataset = LMDataset(
    data_config["root"],
    data_config["train_partition"],
    context_length=data_config["context_length"],
    stride=data_config["stride"],
)
val_dataset = LMDataset(
    data_config["root"],
    data_config["val_partition"],
    context_length=data_config["context_length"],
)
test_dataset = LMDataset(
    data_config["root"],
    data_config["test_partition"],
    context_length=data_config["context_length"],
)

print("Vocabulary size:", tokenizer.vocab_size)
print("Train examples:", len(train_dataset))
print("Validation examples:", len(val_dataset))
print("Test examples:", len(test_dataset))


## Dataloaders


In [ ]:
loader_options = {
    "batch_size": data_config["batch_size"],
    "num_workers": data_config["num_workers"],
    "pin_memory": device == "cuda",
}
train_sampler = EpochWindowSampler(
    train_dataset,
    max_windows=data_config.get("max_train_windows_per_epoch"),
    seed=config["training"]["seed"],
)
train_loader = DataLoader(
    train_dataset,
    sampler=train_sampler,
    **loader_options,
)
val_loader = DataLoader(val_dataset, shuffle=False, **loader_options)
test_loader = DataLoader(test_dataset, shuffle=False, **loader_options)

print("Training windows:", len(train_loader.dataset))
inputs, targets = next(iter(train_loader))
print("Batch shape:", inputs.shape, targets.shape)

print(f"Training windows per epoch: {len(train_sampler):,} / {len(train_dataset):,}")
print(f"Training target tokens per epoch: {len(train_sampler) * data_config['context_length']:,}")
print("Batches (train / val / test):", len(train_loader), len(val_loader), len(test_loader))


## Model


In [ ]:
model = DecoderOnlyTransformer(
    **config["model"],
    max_len=config["data"]["context_length"],
    num_classes=tokenizer.vocab_size,
).to(device)

parameter_count = sum(parameter.numel() for parameter in model.parameters())
print(model)
print("Unique parameters:", f"{parameter_count:,}")


## Trainer

The trainer logs losses, perplexities, learning rate, attention plots, and generated
samples to W&B.


In [ ]:
os.environ.setdefault(
    "HW4P1_OUTPUT_ROOT",
    str(PROJECT_ROOT / "expts"),
)

trainer = LMTrainer(
    model=model,
    tokenizer=tokenizer,
    config=config,
    run_name=config["Name"],
    config_file=str(CONFIG_PATH),
    device=device,
)
trainer.optimizer = create_optimizer(model, config["optimizer"])
trainer.scheduler = create_scheduler(
    trainer.optimizer,
    config["scheduler"],
    train_loader,
    config["training"]["gradient_accumulation_steps"],
)


### Resume from a Checkpoint

Skip this cell for a new run. For a resumed run, replace the path below and run it
before training.


In [ ]:
# trainer.load_checkpoint(str(trainer.last_model_path))


# Train

Run the mini configuration first. For final training, update the config, rebuild the
datasets, model, and trainer, and run this cell again.


In [ ]:
trainer.train(
    train_dataloader=train_loader,
    val_dataloader=val_loader,
    epochs=config["training"]["epochs"],
)


# Evaluate

Run this section after final training and model selection. Load your best checkpoint
before evaluating the test partition.


In [ ]:
trainer.load_checkpoint(str(trainer.best_model_path))
test_metrics, test_generated_results = trainer.evaluate(test_loader)
print(test_metrics)


# Evaluation Check

Confirm that final evaluation completed before generating submission evidence.


In [ ]:
print("Evaluation metrics:", test_metrics)


# Submission Evidence — DO NOT MODIFY

Run this cell once after final evaluation. It creates the required evidence files and
`handin.tar.gz` from the selected checkpoint, trainer history, configuration, test
metrics, and generation results. Do not edit this cell or the generated files.

Take a screenshot of the run page and save it in the repository
root as `training_screenshot.png`. It must show the run name or ID and the complete
training/validation PPL plot. Do not include an API key. The notebook always generates
`training_curve.png` from local trainer history.

This marker does not make a notebook cell tamper-proof. Gradescope will validate the
training results, run metadata, and submitted code independently.


In [ ]:
# ======================== DO NOT MODIFY THIS CELL ========================
from datetime import datetime, timezone
import json
import platform
import shutil
import tarfile
from pathlib import Path

import matplotlib.pyplot as plt
import torch

# Validate required evidence before writing submission files.
if not config["training"].get("use_wandb", False):
    raise RuntimeError("W&B logging is required for this assignment.")

wandb_run = getattr(trainer, "wandb_run", None)
if wandb_run is None or not all(
    isinstance(getattr(wandb_run, key, None), str)
    and getattr(wandb_run, key).strip()
    for key in ("entity", "project", "id")
):
    raise RuntimeError(
        "W&B run information is missing. "
        "Generate submission evidence before calling trainer.cleanup()."
    )

screenshot_source = PROJECT_ROOT / "training_screenshot.png"
if not screenshot_source.is_file():
    raise FileNotFoundError(
        "Save your W&B run screenshot as training_screenshot.png in the project folder, then rerun this cell."
    )

with screenshot_source.open("rb") as screenshot_file:
    if screenshot_file.read(8) != b"\x89PNG\r\n\x1a\n":
        raise ValueError(
            "training_screenshot.png must be a PNG image."
        )

EVIDENCE_SCHEMA_VERSION = 1
EVIDENCE_DIR = PROJECT_ROOT / "submission_evidence"
EVIDENCE_DIR.mkdir(exist_ok=True)


def _json_ready(value):
    if isinstance(value, dict):
        return {str(key): _json_ready(item) for key, item in value.items()}
    if isinstance(value, (list, tuple)):
        return [_json_ready(item) for item in value]
    if isinstance(value, Path):
        return str(value)
    if isinstance(value, torch.Tensor):
        return value.detach().cpu().item() if value.numel() == 1 else value.detach().cpu().tolist()
    if isinstance(value, (str, int, float, bool)) or value is None:
        return value
    if hasattr(value, "item"):
        return value.item()
    return str(value)


def _write_json(filename, payload):
    path = EVIDENCE_DIR / filename
    path.write_text(
        json.dumps(_json_ready(payload), indent=2, sort_keys=True) + "\n",
        encoding="utf-8",
    )
    return path


selected_checkpoint = Path(trainer.best_model_path)

screenshot_source = PROJECT_ROOT / "training_screenshot.png"
wandb_enabled = bool(config["training"].get("use_wandb", False))
if screenshot_source.is_file():
    shutil.copy2(screenshot_source, EVIDENCE_DIR / "training_screenshot.png")

history = _json_ready(trainer.training_history)
validation_records = [
    record for record in history
    if isinstance(record.get("val"), dict)
    and "ce_loss_char" in record["val"]
]
selected_record = min(
    validation_records,
    key=lambda record: record["val"]["ce_loss_char"],
)

partition_manifests = {}
for partition in sorted({
    config["data"]["train_partition"],
    config["data"]["val_partition"],
    config["data"]["test_partition"],
}):
    manifest_path = DATA_ROOT / partition / "manifest.json"
    partition_manifests[partition] = {
        "path": str(manifest_path.relative_to(DATA_ROOT)),
    }

dataset_info_path = DATA_ROOT / "dataset_info.json"
dataset_info = (
    {"path": dataset_info_path.name}
    if dataset_info_path.is_file()
    else None
)
config_file = {"filename": CONFIG_PATH.name}
generated_utc = datetime.now(timezone.utc).isoformat()

selection_metrics_path = _write_json(
    "selection_metrics.json",
    {
        "schema_version": EVIDENCE_SCHEMA_VERSION,
        "run_name": config["Name"],
        "selection_rule": "minimum validation ce_loss_char",
        "selected_epoch": selected_record["epoch"],
        "validation_metrics": selected_record["val"],
        "checkpoint_filename": selected_checkpoint.name,
    },
)
test_metrics_path = _write_json(
    "test_metrics.json",
    {
        "schema_version": EVIDENCE_SCHEMA_VERSION,
        "run_name": config["Name"],
        "checkpoint_filename": selected_checkpoint.name,
        "metrics": test_metrics,
    },
)
generated_samples_path = _write_json(
    "generated_samples.json",
    {
        "schema_version": EVIDENCE_SCHEMA_VERSION,
        "run_name": config["Name"],
        "checkpoint_filename": selected_checkpoint.name,
        "samples_by_strategy": test_generated_results,
    },
)
training_summary_path = _write_json(
    "training_summary.json",
    {
        "schema_version": EVIDENCE_SCHEMA_VERSION,
        "run_name": config["Name"],
        "generated_utc": generated_utc,
        "seed": config["training"]["seed"],
        "epochs_completed": trainer.current_epoch,
        "global_optimizer_steps": trainer.global_optimizer_step,
        "unique_parameters": parameter_count,
        "device": str(device),
        "python_version": platform.python_version(),
        "torch_version": torch.__version__,
        "config_file": config_file,
        "dataset_info": dataset_info,
        "partition_manifests": partition_manifests,
        "training_history": history,
    },
)
checkpoint_manifest_path = _write_json(
    "checkpoint_manifest.json",
    {
        "schema_version": EVIDENCE_SCHEMA_VERSION,
        "filename": selected_checkpoint.name,
        "selected_epoch": selected_record["epoch"],
        "global_optimizer_steps": trainer.global_optimizer_step,
    },
)

wandb_run = getattr(trainer, "wandb_run", None)
wandb_run_path = _write_json(
    "wandb_run.json",
    {
        "schema_version": EVIDENCE_SCHEMA_VERSION,
        "enabled": bool(config["training"].get("use_wandb", False)),
        "entity": getattr(wandb_run, "entity", None),
        "project": getattr(wandb_run, "project", config["training"].get("wandb_project")),
        "run_id": getattr(wandb_run, "id", None),
        "run_name": getattr(wandb_run, "name", config["Name"]),
        "url": getattr(wandb_run, "url", None),
    },
)

evidence_plot_path = EVIDENCE_DIR / "training_curve.png"
figure, axis = plt.subplots(figsize=(9, 5))
for split in ("train", "val", "test"):
    points = [
        (record["epoch"], record[split]["perplexity_char"])
        for record in history
        if isinstance(record.get(split), dict)
        and "perplexity_char" in record[split]
    ]
    if points:
        axis.plot(
            [point[0] for point in points],
            [point[1] for point in points],
            marker="o",
            label=split,
        )
axis.set_title(
    f"{config['Name']} | {parameter_count:,} parameters | "
    f"{trainer.global_optimizer_step:,} optimizer steps"
)
axis.set_xlabel("Epoch")
axis.set_ylabel("Character perplexity")
axis.grid(alpha=0.3)
axis.legend()
figure.tight_layout()
figure.savefig(evidence_plot_path, dpi=160)
plt.show()
plt.close(figure)

shutil.copy2(CONFIG_PATH, EVIDENCE_DIR / "config.yaml")
shutil.copy2(trainer.expt_root / "model_arch.txt", EVIDENCE_DIR / "model_arch.txt")

evidence_files = sorted(
    path for path in EVIDENCE_DIR.iterdir()
    if path.is_file() and path.name != "submission_manifest.json"
)
submission_manifest_path = _write_json(
    "submission_manifest.json",
    {
        "schema_version": EVIDENCE_SCHEMA_VERSION,
        "generated_utc": generated_utc,
        "files": {
            path.name: {}
            for path in evidence_files
        },
    },
)

archive_path = PROJECT_ROOT / "handin.tar.gz"
with tarfile.open(archive_path, "w:gz") as archive:
    for source_dir in (PROJECT_ROOT / "mytorch", PROJECT_ROOT / "hw4lib"):
        archive.add(
            source_dir,
            arcname=f"handin/{source_dir.name}",
            filter=lambda info: None if "__pycache__" in info.name else info,
        )
    for artifact_path in sorted(EVIDENCE_DIR.iterdir()):
        if artifact_path.is_file():
            archive.add(
                artifact_path,
                arcname=f"handin/{artifact_path.name}",
            )

trainer.cleanup()
print("Evidence directory:", EVIDENCE_DIR)
print("Submission archive:", archive_path)
print("Checkpoint weights were not added to the archive.")
# ====================== END DO NOT MODIFY CELL ======================


# Final Submission Check

Run this after `handin.tar.gz` has been created. It reruns the required code tests and validates the training evidence.


In [ ]:
!"{sys.executable}" -m tests.test_hw4p1
